In [1]:
import pandas as pd
from transformers import AutoTokenizer, RobertaModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import torch

import utils
import const
import models

# Read and Prepare Data

In [2]:
relations_df = utils.get_relations()
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE


In [3]:
unique_docids = relations_df['docid'].unique()

docs_df = utils.get_docs(unique_docids)
docs_df.head()

,raw_text,text,sentences
docid,,,
WSJ_20130322_159,Israeli Prime Minister Benjamin Netanyahu <EVE...,Israeli Prime Minister Benjamin Netanyahu [E1]...,[Israeli Prime Minister Benjamin Netanyahu [E1...
nyt_20130322_strange_computer,"Our <TIMEX3 type=""DATE"" value=""PRESENT_REF"" ti...",Our [TIMEX3]digital[/TIMEX3] age is all about ...,[Our [TIMEX3]digital[/TIMEX3] age is all about...
CNN_20130321_821,"Barack Obama would <EVENT class=""OCCURRENCE"" e...",Barack Obama would [E1]make[/E1] a great stand...,[Barack Obama would [E1]make[/E1] a great stan...
nyt_20130321_cyprus,"A Cyprus <EVENT class=""OCCURRENCE"" eid=""e2001""...",A Cyprus [E2001]exit[/E2001] from the euro uni...,[A Cyprus [E2001]exit[/E2001] from the euro un...
bbc_20130322_1353,"Israel's prime minister has <EVENT class=""OCCU...",Israel's prime minister has [E1]apologised[/E1...,[Israel's prime minister has [E1]apologised[/E...


In [4]:
relations_df = utils.create_context_windows(relations_df, docs_df)
relations_df = utils.create_relation_labels(relations_df)
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation,context_window,relation_id
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE,Israeli Prime Minister Benjamin Netanyahu [T1]...,0
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu [T1]...,1
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE,Israeli Prime Minister Benjamin Netanyahu [T1]...,1
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE,Israeli Prime Minister Benjamin Netanyahu [T1]...,0
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu apol...,1


# Prepare Data for Training

In [5]:
tokenizer = models.create_temp_rel_tokenizer()

In [6]:
# 90% train, 5% validation, 5% test (stratified by label)
train_df, temp_df = train_test_split(
    relations_df,
    test_size=0.1,
    random_state=42,
    stratify=relations_df["relation_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["relation_id"]
)

val_df = utils.augment_data(val_df)
test_df = utils.augment_data(test_df)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 6516
Validation: 724
Test: 726


In [7]:
train_loader = utils.create_data_loader(train_df, tokenizer, batch_size=16)
val_loader = utils.create_data_loader(val_df, tokenizer, batch_size=16)
test_loader = utils.create_data_loader(test_df, tokenizer, batch_size=16)

print("num training batches:", len(train_loader))

input_ids shape: torch.Size([6516, 512])
attention_mask shape: torch.Size([6516, 512])
labels shape: torch.Size([6516])
input_ids shape: torch.Size([724, 512])
attention_mask shape: torch.Size([724, 512])
labels shape: torch.Size([724])
input_ids shape: torch.Size([726, 282])
attention_mask shape: torch.Size([726, 282])
labels shape: torch.Size([726])
num training batches: 408


# Model Creation

In [8]:
loss_fn = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)  # (B,) long

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)  # (B, C)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(torch.argmax(logits, dim=-1).cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return total_loss / len(loader.dataset), macro_f1, acc

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return {
        "val_loss": total_loss / len(loader.dataset),
        "macro_f1": macro_f1,
        "accuracy": acc,
    }

In [9]:
# Build model
num_labels = len(const.relation2id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer).to(device)

# Cross-entropy setup
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To dis

# First Stage: Basic Data Set

In [10]:
# Train
epochs = 12
for epoch in range(epochs):
    train_loss, train_f1, train_acc = train_one_epoch(model, train_loader, optimizer, device)

    val_metrics = evaluate(model, val_loader, device)
    val_loss = val_metrics["val_loss"]
    val_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/12 | Train Loss: 0.9223 | Train F1: 0.3328 | Train Acc: 0.6204 | Val Loss: 0.7458 | Val F1: 0.3904 | Val Acc: 0.7313
Epoch 2/12 | Train Loss: 0.6528 | Train F1: 0.4244 | Train Acc: 0.7688 | Val Loss: 0.5961 | Val F1: 0.4267 | Val Acc: 0.7922
Epoch 3/12 | Train Loss: 0.5182 | Train F1: 0.4880 | Train Acc: 0.8101 | Val Loss: 0.5548 | Val F1: 0.5226 | Val Acc: 0.8089
Epoch 4/12 | Train Loss: 0.4099 | Train F1: 0.5610 | Train Acc: 0.8466 | Val Loss: 0.6042 | Val F1: 0.4921 | Val Acc: 0.8006
Epoch 5/12 | Train Loss: 0.3323 | Train F1: 0.6421 | Train Acc: 0.8756 | Val Loss: 0.5469 | Val F1: 0.5346 | Val Acc: 0.8116
Epoch 6/12 | Train Loss: 0.2681 | Train F1: 0.7337 | Train Acc: 0.9049 | Val Loss: 0.6019 | Val F1: 0.5152 | Val Acc: 0.8089
Epoch 7/12 | Train Loss: 0.2081 | Train F1: 0.7857 | Train Acc: 0.9210 | Val Loss: 0.5969 | Val F1: 0.5650 | Val Acc: 0.7978
Epoch 8/12 | Train Loss: 0.1480 | Train F1: 0.8745 | Train Acc: 0.9496 | Val Loss: 0.6664 | Val F1: 0.6146 | Val Acc: 0.8006


# Second Stage: Augmented Data Set

In [11]:
augmented_df = utils.augment_data(train_df, verbose=True)
augmented_loader = utils.create_data_loader(augmented_df, tokenizer, batch_size=16)

print("num training batches:", len(augmented_loader))

Added 6516 swapped rows
New total rows: 13032
input_ids shape: torch.Size([13032, 512])
attention_mask shape: torch.Size([13032, 512])
labels shape: torch.Size([13032])
num training batches: 815


In [12]:
# Train
# epochs = 6
# for epoch in range(epochs):
#     train_loss, train_f1, train_acc = train_one_epoch(model, augmented_loader, optimizer, device)

#     val_metrics = evaluate(model, val_loader, device)
#     val_loss = val_metrics["val_loss"]
#     val_f1 = val_metrics["macro_f1"]
#     val_acc = val_metrics["accuracy"]

#     print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

# Analysis

In [13]:
from sklearn.metrics import confusion_matrix

# Predict on test set
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)
        all_true.append(labels)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).detach().cpu().numpy()

# Confusion matrix (rows=true, cols=pred)
label_ids = [0, 1, 2, 3]
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{const.id2relation[i]}" for i in label_ids],
    columns=[f"pred_{const.id2relation[i]}" for i in label_ids],
)
display(cm_df)

test_metrics = evaluate(model, test_loader, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} | Test F1: {test_metrics['macro_f1']:.4f} | Test Acc: {test_metrics['accuracy']:.4f}")

,pred_VAGUE,pred_BEFORE,pred_AFTER,pred_EQUAL
true_VAGUE,22,32,24,2
true_BEFORE,26,302,20,8
true_AFTER,16,10,214,12
true_EQUAL,2,10,8,6


Test Loss: 0.8305 | Test F1: 0.5501 | Test Acc: 0.7619


# Saving model weights

In [14]:
filename = 'temp_rel_roberta.pt'
torch.save(model.state_dict(), filename)
print(f"Model saved to '{filename}'")

Model saved to 'temp_rel_roberta.pt'


# Test with loaded weights

In [17]:
state_dict = torch.load('./temp_rel_roberta.pt')
tokenizer_test = models.create_temp_rel_tokenizer()

model_test = models.TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer_test)
model_test.load_state_dict(state_dict, strict=False)
model_test = model_test.to(device)
model_test.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model_test.t1_id).any(dim=1)
        has_t2 = (input_ids == model_test.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]
        
        logits = model_test(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)
        all_true.append(labels)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).detach().cpu().numpy()

# Confusion matrix (rows=true, cols=pred)
label_ids = [0, 1, 2, 3]
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{const.id2relation[i]}" for i in label_ids],
    columns=[f"pred_{const.id2relation[i]}" for i in label_ids],
)
display(cm_df)

test_metrics = evaluate(model_test, test_loader, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} | Test F1: {test_metrics['macro_f1']:.4f} | Test Acc: {test_metrics['accuracy']:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,pred_VAGUE,pred_BEFORE,pred_AFTER,pred_EQUAL
true_VAGUE,22,32,24,2
true_BEFORE,26,302,20,8
true_AFTER,16,10,214,12
true_EQUAL,2,10,8,6


Test Loss: 0.8305 | Test F1: 0.5501 | Test Acc: 0.7619
